##Widgets:

In [0]:
dbutils.widgets.text("notebook_name", "")
dbutils.widgets.text("notebook_status", "")
dbutils.widgets.text("execution_date", "")
dbutils.widgets.text("env", "prod")

notebook_name   = dbutils.widgets.get("notebook_name")
notebook_status = dbutils.widgets.get("notebook_status")
execution_date  = dbutils.widgets.get("execution_date")
env             = dbutils.widgets.get("env")

print(f"Notebook: {notebook_name}")
print(f"Status: {notebook_status}")
print(f"Data: {execution_date}")

##Imports e configuração:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime
import requests

storage_account = dbutils.secrets.get(scope='kv-scope', key='adls-storage-account-name')
storage_key     = dbutils.secrets.get(scope='kv-scope', key='adls-storage-key')
logic_app_url   = dbutils.secrets.get(scope='kv-scope', key='logic-app-webhook-url')

OBS_PATH = f'abfss://gold@{storage_account}.dfs.core.windows.net/observability/'

print('Configuração concluída.')

##Try/except com todo o processamento:

In [0]:
try:

    is_error = notebook_status.startswith("ERROR")
    status   = "ERROR" if is_error else "SUCCESS"

    # Registra log na tabela Delta
    log_data = [(
        notebook_name,
        status,
        notebook_status,
        execution_date,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        env
    )]

    schema = StructType([
        StructField("notebook_name",  StringType(), True),
        StructField("status",         StringType(), True),
        StructField("message",        StringType(), True),
        StructField("execution_date", StringType(), True),
        StructField("logged_at",      StringType(), True),
        StructField("env",            StringType(), True),
    ])

    df_log = spark.createDataFrame(log_data, schema)

    df_log.write.format("delta") \
        .mode("append") \
        .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
        .save(OBS_PATH)

    spark.sql("CREATE DATABASE IF NOT EXISTS gold;")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS gold.tb_observability
        USING DELTA LOCATION '{OBS_PATH}'
    """)

    print(f'Log registrado: {status} — {notebook_name}')

    #  Alerta no email se ERROR
    if is_error:
        payload = {
            "notebook_name": notebook_name,
            "status": status,
            "message": notebook_status,
            "execution_date": execution_date,
            "env": env
        }

        response = requests.post(logic_app_url, json=payload)
        print(f'Alerta email enviado. Status: {response.status_code}')

    dbutils.notebook.exit(f"SUCCESS: Observability registrado — {notebook_name} — {status}")

except Exception as e:
    error_msg = str(e)

    if "SUCCESS" in error_msg:
        dbutils.notebook.exit(error_msg)

    print(f'ERRO no Observability: {error_msg}')
    dbutils.notebook.exit(f"ERROR: Observability falhou. Detalhes: {error_msg}")